In [14]:
# Import necessary libraries for data processing, visualization, machine learning, and model evaluation
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve, classification_report
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
import joblib
import warnings
import os
# Load and Preprocess Data
# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

In [15]:


def load_and_preprocess_data(file_path='cardio_train.csv'):
    """
    Load and preprocess the cardio_train.csv dataset.
    - Convert age from days to years.
    - Remove outliers for blood pressure (ap_hi: 80-250, ap_lo: 40-150).
    - Handle missing values with median.
    - Return cleaned DataFrame.
    
    Args:
        file_path (str): Path to the dataset file (default: 'cardio_train.csv')
    Returns:
        pd.DataFrame: Cleaned dataset
    """
    # Load dataset with semicolon separator
    data = pd.read_csv(file_path, sep=';')
    
    # Convert age from days to years for better interpretability
    data['age'] = data['age'] / 365.25
    
    # Drop id column as it is not a predictive feature
    data = data.drop('id', axis=1)
    
    # Remove outliers for systolic (ap_hi) and diastolic (ap_lo) blood pressure
    data = data[(data['ap_hi'].between(80, 250)) & (data['ap_lo'].between(40, 150))]
    
    # Handle missing values by filling with median values
    if data.isnull().sum().any():
        data = data.fillna(data.median())
    
    return data

In [16]:
# Feature Engineering
def engineer_features(data):
    """
    Perform feature engineering on the dataset.
    - Calculate Body Mass Index (BMI) and pulse pressure.
    - Generate synthetic features: median_income, stress, diet_risk.
    - Return DataFrame with new features.
    
    Args:
        data (pd.DataFrame): Input dataset
    Returns:
        pd.DataFrame: Dataset with engineered features
    """
    # Calculate BMI: weight / (height in meters)^2
    data['bmi'] = data['weight'] / ((data['height'] / 100) ** 2)
    
    # Calculate pulse pressure: systolic - diastolic blood pressure
    data['pulse_pressure'] = data['ap_hi'] - data['ap_lo']
    
    # Generate synthetic features for enhanced modeling
    np.random.seed(42)  # Set seed for reproducibility
    # Median income: Random values between $68,000 and $85,000
    income_range = np.random.uniform(68000, 85000, len(data))
    data['median_income'] = income_range
    
    # Stress: Composite feature based on age and systolic blood pressure
    data['stress'] = data['age'] / 100 + data['ap_hi'] / 200
    
    # Diet risk: Higher risk (0.7) if cholesterol > 1, else lower risk (0.3)
    data['diet_risk'] = np.where(data['cholesterol'] > 1, 0.7, 0.3)
    
    
    return data

In [17]:
def prepare_data(data, selected_features):
    """
    Prepare data for machine learning.
    - Select features and target variable.
    - Plot and save correlation matrix to outputs/ directory.
    - Apply SMOTE to handle class imbalance.
    - Split and scale data.
    - Return train/test sets and scaler.
    
    Args:
        data (pd.DataFrame): Input dataset
        selected_features (list): List of feature names to use
    Returns:
        tuple: (X_train, X_test, y_train, y_test, scaler)
    """
    # Create outputs directory if it doesn't exist
    os.makedirs('../outputs', exist_ok=True)
    
    # Select features and target variable
    X = data[selected_features]
    y = data['cardio']
    
    # Visualize feature correlations and save plot
    plt.figure(figsize=(10, 8))
    sns.heatmap(X.corr(), annot=True, cmap='coolwarm', fmt='.2f')
    plt.title('Correlation Matrix of Features')
    plt.savefig('../outputs/correlation_matrix.png')
    plt.close()
    
    # Handle class imbalance using SMOTE
    smote = SMOTE(random_state=42)
    X, y = smote.fit_resample(X, y)
    
    # Split data into training (80%) and testing (20%) sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Scale features to standardize them
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    return X_train, X_test, y_train, y_test, scaler

In [18]:
# Train Random Forest Model
def train_random_forest(X_train, y_train):
    """
    Train a Random Forest Classifier.
    
    Args:
        X_train (np.array): Training features
        y_train (np.array): Training labels
    Returns:
        RandomForestClassifier: Trained model
    """
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    return model

In [19]:
# Train XGBoost Model
def train_xgboost(X_train, y_train):
    """
    Train an XGBoost Classifier with hyperparameter tuning using GridSearchCV.
    
    Args:
        X_train (np.array): Training features
        y_train (np.array): Training labels
    Returns:
        XGBClassifier: Trained model with best parameters
    """
    model = XGBClassifier( eval_metric='logloss', random_state=42,
                          learning_rate=0.1, max_depth=5, n_estimators=200)
    param_grid = {
        'learning_rate': [0.01, 0.2, 0.3],
        'max_depth': [3, 5, 7],
        'n_estimators': [100, 200, 300]
    }
    grid_search = GridSearchCV(model, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
    grid_search.fit(X_train, y_train)
    print(f"Best XGBoost Parameters: {grid_search.best_params_}")
    return grid_search.best_estimator_

In [20]:
#  Train LightGBM Model
def train_lightgbm(X_train, y_train):
    """
    Train a LightGBM Classifier.
    
    Args:
        X_train (np.array): Training features
        y_train (np.array): Training labels
    Returns:
        LGBMClassifier: Trained model
    """
    model = LGBMClassifier(random_state=42)
    model.fit(X_train, y_train)
    return model

In [21]:
# Train Decision Tree Model
def train_decision_tree(X_train, y_train):
    """
    Train a Decision Tree Classifier.
    
    Args:
        X_train (np.array): Training features
        y_train (np.array): Training labels
    Returns:
        DecisionTreeClassifier: Trained model
    """
    model = DecisionTreeClassifier(random_state=42)
    model.fit(X_train, y_train)
    return model

In [22]:
# Cell 9: Train KNN Model
def train_knn(X_train, y_train):
    """
    Train a K-Nearest Neighbors Classifier.
    
    Args:
        X_train (np.array): Training features
        y_train (np.array): Training labels
    Returns:
        KNeighborsClassifier: Trained model
    """
    model = KNeighborsClassifier()
    model.fit(X_train, y_train)
    return model

In [23]:
# Cell 10: Train Logistic Regression Model
def train_logistic_regression(X_train, y_train):
    """
    Train a Logistic Regression Classifier.
    
    Args:
        X_train (np.array): Training features
        y_train (np.array): Training labels
    Returns:
        LogisticRegression: Trained model
    """
    model = LogisticRegression(random_state=42)
    model.fit(X_train, y_train)
    return model

In [24]:
def evaluate_and_plot(model, X_test, y_test, model_name, selected_features=None):
    """
    Evaluate model performance and generate visualizations.
    - Compute metrics (accuracy, precision, recall, F1, AUC-ROC) with multiple thresholds.
    - Plot and save confusion matrix and ROC curve data to outputs/ directory.
    - Plot feature importance for Random Forest and XGBoost.
    
    Args:
        model: Trained machine learning model
        X_test (np.array): Test features
        y_test (np.array): Test labels
        model_name (str): Name of the model
        selected_features (list): List of feature names (for feature importance)
    Returns:
        tuple: (metrics, (fpr, tpr)) - Evaluation metrics and ROC curve data
    """
    # Ensure outputs directory exists
    os.makedirs('../outputs', exist_ok=True)
    
    # Predict probabilities for ROC curve and threshold evaluation
    y_prob = model.predict_proba(X_test)[:, 1]
    
    # Evaluate different decision thresholds to optimize accuracy
    thresholds = [0.3, 0.4, 0.5, 0.6]
    best_accuracy = 0
    best_threshold = 0.5
    best_y_pred = model.predict(X_test)
    
    for threshold in thresholds:
        y_pred = (y_prob >= threshold).astype(int)
        accuracy = accuracy_score(y_test, y_pred)
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_threshold = threshold
            best_y_pred = y_pred
    
    # Compute evaluation metrics with the best threshold
    metrics = {
        'accuracy': accuracy_score(y_test, best_y_pred),
        'precision': precision_score(y_test, best_y_pred),
        'recall': recall_score(y_test, best_y_pred),
        'f1': f1_score(y_test, best_y_pred),
        'auc': roc_auc_score(y_test, y_prob)
    }
    
    # Print evaluation metrics and classification report
    print(f"\n{model_name} Evaluation (Best Threshold: {best_threshold}):")
    for metric, value in metrics.items():
        print(f"{metric.capitalize()}: {value:.4f}")
    print("Classification Report:")
    print(classification_report(y_test, best_y_pred))
    
    # Plot and save confusion matrix
    cm = confusion_matrix(y_test, best_y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - {model_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f'../outputs/confusion_matrix_{model_name.lower()}.png')
    plt.close()
    
    # Compute ROC curve data
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    
    # Plot feature importance for Random Forest and XGBoost
    if model_name in ['RandomForest', 'XGBoost'] and selected_features is not None:
        importance = model.feature_importances_ if model_name == 'RandomForest' else model.feature_importances_
        feature_importance = pd.DataFrame({
            'feature': selected_features,
            'importance': importance
        }).sort_values('importance', ascending=False)
        print(f"\n{model_name} Feature Importance:")
        print(feature_importance)
        
        plt.figure(figsize=(10, 6))
        sns.barplot(x='importance', y='feature', data=feature_importance)
        plt.title(f'{model_name} Feature Importance')
        plt.savefig(f'../outputs/feature_importance_{model_name.lower()}.png')
        plt.close()
    
    return metrics, (fpr, tpr)

In [25]:
def main():
    """
    Orchestrate the machine learning pipeline.
    - Load and preprocess data.
    - Engineer features.
    - Prepare data for modeling.
    - Train multiple models.
    - Evaluate models and generate plots.
    - Save models, scaler, and plots.
    """
    # Create models directory if it doesn't exist
    os.makedirs('models', exist_ok=True)
    
    # Define features to use in modeling
    selected_features = [
        'age', 'gender', 'cholesterol', 'gluc', 'smoke', 'alco', 'active',
        'bmi', 'pulse_pressure', 'median_income', 'stress', 'diet_risk'
    ]
    
    # Load and preprocess the dataset
    data = load_and_preprocess_data()
    
    # Perform feature engineering
    data = engineer_features(data)
    
    # Prepare data for modeling (split, scale, balance)
    X_train, X_test, y_train, y_test, scaler = prepare_data(data, selected_features)
    
    # Define models and their corresponding training functions
    model_functions = {
        'RandomForest': train_random_forest,
        'XGBoost': train_xgboost,
        'LightGBM': train_lightgbm,
        'DecisionTree': train_decision_tree,
        'KNN': train_knn,
        'LogisticRegression': train_logistic_regression
    }
    
    # Train and evaluate each model
    results = {}
    roc_curves = {}
    for name, train_func in model_functions.items():
        # Train the model
        model = train_func(X_train, y_train)
        
        # Evaluate the model and generate plots
        metrics, roc_data = evaluate_and_plot(model, X_test, y_test, name, selected_features)
        results[name] = metrics
        roc_curves[name] = roc_data
        
        # Save the trained model
        joblib.dump(model, f'../models/{name}_model.pkl')
    
    # Save the scaler
    joblib.dump(scaler, '../models/scaler.pkl')
    
    # Plot ROC curves for all models in a single figure
    plt.figure(figsize=(8, 6))
    for name, (fpr, tpr) in roc_curves.items():
        plt.plot(fpr, tpr, label=f'{name} (AUC = {results[name]["auc"]:.2f})')
    plt.plot([0, 1], [0, 1], 'k--')  # Diagonal line for random guessing
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves')
    plt.legend()
    plt.savefig('../outputs/roc_curves.png')
    plt.close()
    
    print("\nAll tasks completed successfully.")

In [26]:
# Cell 13: Execute Main Function
if __name__ == '__main__':
    main()


RandomForest Evaluation (Best Threshold: 0.5):
Accuracy: 0.7209
Precision: 0.7261
Recall: 0.7109
F1: 0.7184
Auc: 0.7826
Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.73      0.72      6933
           1       0.73      0.71      0.72      6955

    accuracy                           0.72     13888
   macro avg       0.72      0.72      0.72     13888
weighted avg       0.72      0.72      0.72     13888


RandomForest Feature Importance:
           feature  importance
10          stress    0.286923
0              age    0.190807
7              bmi    0.172425
9    median_income    0.166841
8   pulse_pressure    0.091393
2      cholesterol    0.021218
11       diet_risk    0.016808
3             gluc    0.014434
1           gender    0.014025
6           active    0.011428
4            smoke    0.007331
5             alco    0.006367
Best XGBoost Parameters: {'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 100}

XGBoos